In [ ]:
from dotenv import load_dotenv
import os 
import pandas as pd
import requests
import datetime as dt
import matplotlib.pyplot as plt
import numpy as np

load_dotenv()

API_KEY = os.getenv('FMP_API_KEY')
API_BASE = "https://financialmodelingprep.com/stable"

SESSION = requests.Session()
SESSION.headers.update({"Accept-Encoding": "gzip, deflate"}) 

TICKER = "JPY=X"
DATE_START = pd.to_datetime("2016-01-01").date().isoformat()
DATE_END   = dt.datetime.now().date().isoformat()
BB_LEN = 20
DEVS = 2

In [52]:
def fmp_get(
        endpoint,
        params, 

):
    params={
        **params,
        "apikey": API_KEY,
    }
    url = f"{API_BASE}/{endpoint}"
    r = SESSION.get(url, params=params)
    return r.json()

In [53]:
def get_prices(
        ticker,
        start, 
        end
):
    data = fmp_get(
        endpoint="historical-price-eod/full",
        params={
            "symbol": ticker,
            "from": start,
            "to": end
        }
    )

    df = pd.DataFrame(data).sort_values('date').reset_index(drop=True)
    return df

In [54]:
def add_bollinger_bands(df, devs=DEVS, bb_len=BB_LEN):

    # can change to ema (use MACD video/code for reference)
    df['BB_SMA'] = df['close'].rolling(bb_len).mean()

    # get the standard deviation of the close prices for the period
    df['BB_STD'] = df['close'].rolling(bb_len).std()

    df['Upper_Band'] = df['BB_SMA'] + (devs * df['BB_STD'])
    df['Lower_Band'] = df['BB_SMA'] - (devs * df['BB_STD'])

    df = df.dropna()

    plt.figure()
    plt.plot(df['close'], color='blue')
    plt.plot(df['Upper_Band'], color='orange')
    plt.plot(df['Lower_Band'], color='orange')

    plt.title(f'{TICKER} Bollinger Bands. Len: {BB_LEN}, Deviations: {DEVS}');

    return df

def add_BB_strategy(df):
    df['BB_Strategy'] = 0
    df['BB_Strategy'] = np.where(
        df['close'] > df['Upper_Band'], -1, 
        np.where(df['close'] < df['Lower_Band'], 1, 0)
        )
    
    df['BB_Strategy'] = df['BB_Strategy'].shift(1)
    
    return df

def add_VWAP_strategy(df):
    df['VWAP_Strategy'] = 0
    df['VWAP_Strategy'] = np.where(
        df['close'] < df['vwap'], 1, 
        np.where(df['close'] > df['vwap'], -1, 0)
        )
    
    df['VWAP_Strategy'] = df['VWAP_Strategy'].shift(1)
    
    return df

def add_full_strategy(df):

    df['Full_Strategy'] = df['VWAP_Strategy'] + df['BB_Strategy']

    # adjust values for simplicity
    df['Strategy'] = np.where(df['Full_Strategy'] == 2, 1, 
                     np.where(df['Full_Strategy'] == -2, -1, 0))

    return df

def test_strategy(df):

    df['Asset_Returns'] = (1 + df['close'].pct_change()).cumprod() - 1
    df['Strategy_Returns'] = (1 + df['close'].pct_change() * df['Strategy']).cumprod() - 1

    # plot the strategy returns
    plt.figure()
    plt.plot(df['Asset_Returns'])
    plt.plot(df['Strategy_Returns'])
    plt.legend([f'{TICKER} Cumulative', 'Strategy Cumulative'])
    plt.title(F'VWAP Strategy vs {TICKER}');

    return df

In [55]:
def main():
    df = get_prices(
        ticker=TICKER,
        start=DATE_START,
        end=DATE_END
    )
    plt.plot(df.close)
    plt.plot(df.vwap)
    plt.title(f"Price of {TICKER} and VWAP")
    plt.legend([
        f'{TICKER} Price',
        f'{TICKER} VWAP'
    ])

    df = add_bollinger_bands(df)
    df = add_BB_strategy(df)
    df = add_VWAP_strategy(df)
    df = add_full_strategy(df)
    df = test_strategy(df)

    return df

df = main()

KeyError: 'date'